In [1]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("hospital").getOrCreate()
sc=spark.sparkContext
print("spark is created")

spark is created


In [2]:
patients_csv = """patient_id,name,city,age,gender,registration_date
1,Aarav,Hyderabad,29,Male,2023-01-10
2,Priya,Bangalore,34,Female,2023-02-12
3,Rahul,Mumbai,41,Male,2023-03-14
4,Sneha,Delhi,26,Female,2023-04-15
5,Kiran,Chennai,37,Male,2023-05-11
6,Meera,Hyderabad,31,Female,2023-06-10
7,Amit,Pune,45,Male,2023-06-22
8,Neha,Delhi,28,Female,2023-07-10
9,Divya,Bangalore,33,Female,2023-07-15
10,Vikram,Mumbai,52,Male,2023-08-01
11,Farhan,Hyderabad,39,Male,2023-08-10
12,Simran,Delhi,25,Female,2023-08-21
"""

In [3]:
doctors_csv = """doctor_id,doctor_name,specialization,city,consultation_fee
101,Dr Sharma,Cardiology,Hyderabad,1200
102,Dr Iyer,Dermatology,Bangalore,800
103,Dr Khan,Orthopedics,Mumbai,1500
104,Dr Reddy,Pediatrics,Delhi,900
105,Dr Mehta,Neurology,Hyderabad,2000
106,Dr Nair,Cardiology,Chennai,1300
107,Dr Verma,Dermatology,Pune,850
108,Dr Rao,Orthopedics,Delhi,1400
"""

In [4]:
appointments_csv = """appointment_id,patient_id,doctor_id,appointment_date,status
1,1,101,2024-03-01,Completed
2,2,102,2024-03-01,Completed
3,3,103,2024-03-02,Completed
4,4,104,2024-03-02,Pending
5,5,106,2024-03-03,Completed
6,6,105,2024-03-03,Completed
7,7,107,2024-03-04,Cancelled
8,8,108,2024-03-04,Completed
9,9,102,2024-03-05,Completed
10,10,103,2024-03-05,Completed
11,11,101,2024-03-06,Pending
12,12,104,2024-03-06,Completed
13,1,105,2024-03-07,Completed
14,3,108,2024-03-07,Completed
15,6,101,2024-03-08,Cancelled
16,9,106,2024-03-08,Completed
"""

In [5]:
bills_csv = """bill_id,appointment_id,bill_amount,payment_mode,payment_status
1,1,1200,UPI,Paid
2,2,800,Credit Card,Paid
3,3,1500,Cash,Paid
4,4,900,UPI,Pending
5,5,1300,Debit Card,Paid
6,6,2000,Credit Card,Paid
7,7,850,Cash,Cancelled
8,8,1400,UPI,Paid
9,9,800,UPI,Paid
10,10,1500,Credit Card,Paid
11,11,1200,UPI,Pending
12,12,900,Cash,Paid
13,13,2000,Credit Card,Paid
14,14,1400,UPI,Paid
15,15,1200,Cash,Cancelled
16,16,1300,Debit Card,Paid
"""

In [6]:
hospital_logs_txt = """Aarav login
Priya login
Rahul appointment
Sneha login
Aarav payment
Kiran appointment
Meera login
Vikram appointment
Divya payment
Farhan login
Simran appointment
Neha payment
Amit login
Rahul payment
Meera appointment
Aarav logout
Priya payment
Divya login
Vikram payment
Farhan appointment
"""


In [7]:
patient_profiles_json = """[
  {
    "patient_id": 1,
    "name": "Aarav",
    "contact": {"email": "aarav@mail.com", "phone": "9000011111"},
    "allergies": ["Dust", "Peanuts"]
  },
  {
    "patient_id": 2,
    "name": "Priya",
    "contact": {"email": "priya@mail.com", "phone": "9000022222"},
    "allergies": ["Pollen"]
  },
  {
    "patient_id": 3,
    "name": "Rahul",
    "contact": {"email": "rahul@mail.com", "phone": "9000033333"},
    "allergies": ["Dust", "Milk"]
  },
  {
    "patient_id": 6,
    "name": "Meera",
    "contact": {"email": "meera@mail.com", "phone": "9000066666"},
    "allergies": ["Seafood"]
  },
  {
    "patient_id": 10,
    "name": "Vikram",
    "contact": {"email": "vikram@mail.com", "phone": "9000101010"},
    "allergies": ["Pollen", "Dust"]
  }
]
"""


In [8]:
with open("patients.csv", "w") as f:
    f.write(patients_csv)

with open("doctors.csv", "w") as f:
    f.write(doctors_csv)

with open("appointments.csv", "w") as f:
    f.write(appointments_csv)

with open("bills.csv", "w") as f:
    f.write(bills_csv)

with open("hospital_logs.txt", "w") as f:
    f.write(hospital_logs_txt)

with open("patient_profiles.json", "w") as f:
    f.write(patient_profiles_json)

print("Hospital datasets created successfully")

Hospital datasets created successfully


In [9]:
patients = spark.read.csv("patients.csv", header=True, inferSchema=True)
doctors = spark.read.csv("doctors.csv", header=True, inferSchema=True)
appointments = spark.read.csv("appointments.csv", header=True, inferSchema=True)
bills = spark.read.csv("bills.csv", header=True, inferSchema=True)
profiles = spark.read.json("patient_profiles.json", multiLine=True)

Section 1 — DataFrame Basics

In [10]:
patients.show()
doctors.show()
appointments.show()
bills.show()

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|
+----------+------+---------+---+------+-----------------+

+---------+-----------+--------------+---------+-------

In [11]:
patients.printSchema()
doctors.printSchema()
appointments.printSchema()

root
 |-- patient_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- registration_date: date (nullable = true)

root
 |-- doctor_id: integer (nullable = true)
 |-- doctor_name: string (nullable = true)
 |-- specialization: string (nullable = true)
 |-- city: string (nullable = true)
 |-- consultation_fee: integer (nullable = true)

root
 |-- appointment_id: integer (nullable = true)
 |-- patient_id: integer (nullable = true)
 |-- doctor_id: integer (nullable = true)
 |-- appointment_date: date (nullable = true)
 |-- status: string (nullable = true)



In [12]:
print("total patients:",patients.count())
print("total doctors:",doctors.count())
appointments.show(5)

total patients: 12
total doctors: 8
+--------------+----------+---------+----------------+---------+
|appointment_id|patient_id|doctor_id|appointment_date|   status|
+--------------+----------+---------+----------------+---------+
|             1|         1|      101|      2024-03-01|Completed|
|             2|         2|      102|      2024-03-01|Completed|
|             3|         3|      103|      2024-03-02|Completed|
|             4|         4|      104|      2024-03-02|  Pending|
|             5|         5|      106|      2024-03-03|Completed|
+--------------+----------+---------+----------------+---------+
only showing top 5 rows


Section 2 — Select, Rename, Filter

In [13]:
patients.select("name","city","age").show()

+------+---------+---+
|  name|     city|age|
+------+---------+---+
| Aarav|Hyderabad| 29|
| Priya|Bangalore| 34|
| Rahul|   Mumbai| 41|
| Sneha|    Delhi| 26|
| Kiran|  Chennai| 37|
| Meera|Hyderabad| 31|
|  Amit|     Pune| 45|
|  Neha|    Delhi| 28|
| Divya|Bangalore| 33|
|Vikram|   Mumbai| 52|
|Farhan|Hyderabad| 39|
|Simran|    Delhi| 25|
+------+---------+---+



In [14]:
doctors.select("doctor_name","specialization","consultation_fee").show()

+-----------+--------------+----------------+
|doctor_name|specialization|consultation_fee|
+-----------+--------------+----------------+
|  Dr Sharma|    Cardiology|            1200|
|    Dr Iyer|   Dermatology|             800|
|    Dr Khan|   Orthopedics|            1500|
|   Dr Reddy|    Pediatrics|             900|
|   Dr Mehta|     Neurology|            2000|
|    Dr Nair|    Cardiology|            1300|
|   Dr Verma|   Dermatology|             850|
|     Dr Rao|   Orthopedics|            1400|
+-----------+--------------+----------------+



In [18]:
from pyspark.sql.functions import col
patients.withColumnRenamed("name","patient_name").show()

+----------+------------+---------+---+------+-----------------+
|patient_id|patient_name|     city|age|gender|registration_date|
+----------+------------+---------+---+------+-----------------+
|         1|       Aarav|Hyderabad| 29|  Male|       2023-01-10|
|         2|       Priya|Bangalore| 34|Female|       2023-02-12|
|         3|       Rahul|   Mumbai| 41|  Male|       2023-03-14|
|         4|       Sneha|    Delhi| 26|Female|       2023-04-15|
|         5|       Kiran|  Chennai| 37|  Male|       2023-05-11|
|         6|       Meera|Hyderabad| 31|Female|       2023-06-10|
|         7|        Amit|     Pune| 45|  Male|       2023-06-22|
|         8|        Neha|    Delhi| 28|Female|       2023-07-10|
|         9|       Divya|Bangalore| 33|Female|       2023-07-15|
|        10|      Vikram|   Mumbai| 52|  Male|       2023-08-01|
|        11|      Farhan|Hyderabad| 39|  Male|       2023-08-10|
|        12|      Simran|    Delhi| 25|Female|       2023-08-21|
+----------+------------+

In [19]:
doctors.withColumnRenamed("doctor_name","consultant_name").show()

+---------+---------------+--------------+---------+----------------+
|doctor_id|consultant_name|specialization|     city|consultation_fee|
+---------+---------------+--------------+---------+----------------+
|      101|      Dr Sharma|    Cardiology|Hyderabad|            1200|
|      102|        Dr Iyer|   Dermatology|Bangalore|             800|
|      103|        Dr Khan|   Orthopedics|   Mumbai|            1500|
|      104|       Dr Reddy|    Pediatrics|    Delhi|             900|
|      105|       Dr Mehta|     Neurology|Hyderabad|            2000|
|      106|        Dr Nair|    Cardiology|  Chennai|            1300|
|      107|       Dr Verma|   Dermatology|     Pune|             850|
|      108|         Dr Rao|   Orthopedics|    Delhi|            1400|
+---------+---------------+--------------+---------+----------------+



In [21]:
patients.filter("city=='Hyderabad'").show()

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
+----------+------+---------+---+------+-----------------+



In [22]:
patients.filter("gender='Female'").show()

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|         2| Priya|Bangalore| 34|Female|       2023-02-12|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|
+----------+------+---------+---+------+-----------------+



In [24]:
patients.filter("age > 35").show()

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
+----------+------+---------+---+------+-----------------+



In [25]:
doctors.filter("city=='Hyderabad'").show()

+---------+-----------+--------------+---------+----------------+
|doctor_id|doctor_name|specialization|     city|consultation_fee|
+---------+-----------+--------------+---------+----------------+
|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|
|      105|   Dr Mehta|     Neurology|Hyderabad|            2000|
+---------+-----------+--------------+---------+----------------+



In [27]:
doctors.filter("specialization=='Cardiology'").show()

+---------+-----------+--------------+---------+----------------+
|doctor_id|doctor_name|specialization|     city|consultation_fee|
+---------+-----------+--------------+---------+----------------+
|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|
|      106|    Dr Nair|    Cardiology|  Chennai|            1300|
+---------+-----------+--------------+---------+----------------+



In [28]:
doctors.filter("consultation_fee>1000").show()

+---------+-----------+--------------+---------+----------------+
|doctor_id|doctor_name|specialization|     city|consultation_fee|
+---------+-----------+--------------+---------+----------------+
|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|
|      103|    Dr Khan|   Orthopedics|   Mumbai|            1500|
|      105|   Dr Mehta|     Neurology|Hyderabad|            2000|
|      106|    Dr Nair|    Cardiology|  Chennai|            1300|
|      108|     Dr Rao|   Orthopedics|    Delhi|            1400|
+---------+-----------+--------------+---------+----------------+



Section 3 — Sorting and Limit

In [30]:
patients.orderBy("age").show()

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|        12|Simran|    Delhi| 25|Female|       2023-08-21|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|
+----------+------+---------+---+------+-----------------+



In [31]:
patients.orderBy(col("age").desc()).show()

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|
+----------+------+---------+---+------+-----------------+



In [32]:
patients.orderBy(col("age").desc()).show(5)

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|
+----------+------+---------+---+------+-----------------+
only showing top 5 rows


In [33]:
patients.orderBy("age").show(3)

+----------+------+-----+---+------+-----------------+
|patient_id|  name| city|age|gender|registration_date|
+----------+------+-----+---+------+-----------------+
|        12|Simran|Delhi| 25|Female|       2023-08-21|
|         4| Sneha|Delhi| 26|Female|       2023-04-15|
|         8|  Neha|Delhi| 28|Female|       2023-07-10|
+----------+------+-----+---+------+-----------------+
only showing top 3 rows


In [34]:
doctors.orderBy(col("consultation_fee").desc()).show()

+---------+-----------+--------------+---------+----------------+
|doctor_id|doctor_name|specialization|     city|consultation_fee|
+---------+-----------+--------------+---------+----------------+
|      105|   Dr Mehta|     Neurology|Hyderabad|            2000|
|      103|    Dr Khan|   Orthopedics|   Mumbai|            1500|
|      108|     Dr Rao|   Orthopedics|    Delhi|            1400|
|      106|    Dr Nair|    Cardiology|  Chennai|            1300|
|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|
|      104|   Dr Reddy|    Pediatrics|    Delhi|             900|
|      107|   Dr Verma|   Dermatology|     Pune|             850|
|      102|    Dr Iyer|   Dermatology|Bangalore|             800|
+---------+-----------+--------------+---------+----------------+



In [35]:
doctors.orderBy(col("consultation_fee").desc()).show(3)

+---------+-----------+--------------+---------+----------------+
|doctor_id|doctor_name|specialization|     city|consultation_fee|
+---------+-----------+--------------+---------+----------------+
|      105|   Dr Mehta|     Neurology|Hyderabad|            2000|
|      103|    Dr Khan|   Orthopedics|   Mumbai|            1500|
|      108|     Dr Rao|   Orthopedics|    Delhi|            1400|
+---------+-----------+--------------+---------+----------------+
only showing top 3 rows


In [36]:
doctors.orderBy(col("consultation_fee")).show(3)

+---------+-----------+--------------+---------+----------------+
|doctor_id|doctor_name|specialization|     city|consultation_fee|
+---------+-----------+--------------+---------+----------------+
|      102|    Dr Iyer|   Dermatology|Bangalore|             800|
|      107|   Dr Verma|   Dermatology|     Pune|             850|
|      104|   Dr Reddy|    Pediatrics|    Delhi|             900|
+---------+-----------+--------------+---------+----------------+
only showing top 3 rows


In [37]:
appointments.orderBy("appointment_date").show()

+--------------+----------+---------+----------------+---------+
|appointment_id|patient_id|doctor_id|appointment_date|   status|
+--------------+----------+---------+----------------+---------+
|             1|         1|      101|      2024-03-01|Completed|
|             2|         2|      102|      2024-03-01|Completed|
|             3|         3|      103|      2024-03-02|Completed|
|             4|         4|      104|      2024-03-02|  Pending|
|             5|         5|      106|      2024-03-03|Completed|
|             6|         6|      105|      2024-03-03|Completed|
|             7|         7|      107|      2024-03-04|Cancelled|
|             8|         8|      108|      2024-03-04|Completed|
|             9|         9|      102|      2024-03-05|Completed|
|            10|        10|      103|      2024-03-05|Completed|
|            11|        11|      101|      2024-03-06|  Pending|
|            12|        12|      104|      2024-03-06|Completed|
|            13|         

In [38]:
bills.orderBy(col("bill_amount").desc()).show()

+-------+--------------+-----------+------------+--------------+
|bill_id|appointment_id|bill_amount|payment_mode|payment_status|
+-------+--------------+-----------+------------+--------------+
|      6|             6|       2000| Credit Card|          Paid|
|     13|            13|       2000| Credit Card|          Paid|
|      3|             3|       1500|        Cash|          Paid|
|     10|            10|       1500| Credit Card|          Paid|
|      8|             8|       1400|         UPI|          Paid|
|     14|            14|       1400|         UPI|          Paid|
|      5|             5|       1300|  Debit Card|          Paid|
|     16|            16|       1300|  Debit Card|          Paid|
|      1|             1|       1200|         UPI|          Paid|
|     11|            11|       1200|         UPI|       Pending|
|     15|            15|       1200|        Cash|     Cancelled|
|      4|             4|        900|         UPI|       Pending|
|     12|            12| 

In [39]:
bills.orderBy(col("bill_amount").desc()).show(5)

+-------+--------------+-----------+------------+--------------+
|bill_id|appointment_id|bill_amount|payment_mode|payment_status|
+-------+--------------+-----------+------------+--------------+
|     13|            13|       2000| Credit Card|          Paid|
|      6|             6|       2000| Credit Card|          Paid|
|      3|             3|       1500|        Cash|          Paid|
|     10|            10|       1500| Credit Card|          Paid|
|      8|             8|       1400|         UPI|          Paid|
+-------+--------------+-----------+------------+--------------+
only showing top 5 rows


Section 4 — Aggregations

In [43]:
from pyspark.sql.functions import count, sum, avg, min, max
patients.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|  Chennai|    1|
|   Mumbai|    2|
|     Pune|    1|
|    Delhi|    3|
|Hyderabad|    3|
+---------+-----+



In [44]:
patients.groupBy("gender").count().show()

+------+-----+
|gender|count|
+------+-----+
|Female|    6|
|  Male|    6|
+------+-----+



In [45]:
doctors.groupBy("specialization").count().show()

+--------------+-----+
|specialization|count|
+--------------+-----+
|     Neurology|    1|
|   Dermatology|    2|
|    Cardiology|    2|
|    Pediatrics|    1|
|   Orthopedics|    2|
+--------------+-----+



In [47]:
patients.select(avg("age")).show()

+--------+
|avg(age)|
+--------+
|    35.0|
+--------+



In [48]:
patients.select(max("age")).show()
patients.select(min("age")).show()

+--------+
|max(age)|
+--------+
|      52|
+--------+

+--------+
|min(age)|
+--------+
|      25|
+--------+



In [49]:
doctors.select(avg("consultation_fee")).show()

+---------------------+
|avg(consultation_fee)|
+---------------------+
|              1243.75|
+---------------------+



In [51]:
doctors.select(max("consultation_fee")).show()

+---------------------+
|max(consultation_fee)|
+---------------------+
|                 2000|
+---------------------+



In [53]:
bills.select(sum("bill_amount")).show()

+----------------+
|sum(bill_amount)|
+----------------+
|           20250|
+----------------+



In [54]:
bills.select(avg("bill_amount")).show()

+----------------+
|avg(bill_amount)|
+----------------+
|        1265.625|
+----------------+



Section 5 — GroupBy Analytics

In [55]:
patients.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|  Chennai|    1|
|   Mumbai|    2|
|     Pune|    1|
|    Delhi|    3|
|Hyderabad|    3|
+---------+-----+



In [56]:
doctors.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    1|
|  Chennai|    1|
|   Mumbai|    1|
|     Pune|    1|
|    Delhi|    2|
|Hyderabad|    2|
+---------+-----+



In [57]:
appointments.groupBy("status").count().show()

+---------+-----+
|   status|count|
+---------+-----+
|Completed|   12|
|Cancelled|    2|
|  Pending|    2|
+---------+-----+



In [58]:
bills.groupBy("payment_status").sum("bill_amount").show()

+--------------+----------------+
|payment_status|sum(bill_amount)|
+--------------+----------------+
|     Cancelled|            2050|
|       Pending|            2100|
|          Paid|           16100|
+--------------+----------------+



In [59]:
bills.groupBy("payment_mode").sum("bill_amount").show()

+------------+----------------+
|payment_mode|sum(bill_amount)|
+------------+----------------+
| Credit Card|            6300|
|        Cash|            4450|
|  Debit Card|            2600|
|         UPI|            6900|
+------------+----------------+



In [60]:
bills.groupBy("payment_mode").avg("bill_amount").show()

+------------+----------------+
|payment_mode|avg(bill_amount)|
+------------+----------------+
| Credit Card|          1575.0|
|        Cash|          1112.5|
|  Debit Card|          1300.0|
|         UPI|          1150.0|
+------------+----------------+



In [61]:
appointments.groupBy("doctor_id").count().show()

+---------+-----+
|doctor_id|count|
+---------+-----+
|      108|    2|
|      101|    3|
|      103|    2|
|      107|    1|
|      102|    2|
|      105|    2|
|      106|    2|
|      104|    2|
+---------+-----+



In [62]:
appointments.groupBy("patient_id").count().show()

+----------+-----+
|patient_id|count|
+----------+-----+
|        12|    1|
|         1|    2|
|         6|    2|
|         3|    2|
|         5|    1|
|         9|    2|
|         4|    1|
|         8|    1|
|         7|    1|
|        10|    1|
|        11|    1|
|         2|    1|
+----------+-----+



In [63]:
bills.groupBy("appointment_id").sum("bill_amount").show()

+--------------+----------------+
|appointment_id|sum(bill_amount)|
+--------------+----------------+
|            12|             900|
|             1|            1200|
|            13|            2000|
|             6|            2000|
|            16|            1300|
|             3|            1500|
|             5|            1300|
|            15|            1200|
|             9|             800|
|             4|             900|
|             8|            1400|
|             7|             850|
|            10|            1500|
|            11|            1200|
|            14|            1400|
|             2|             800|
+--------------+----------------+



In [64]:
doctors.groupBy("specialization").avg("consultation_fee").show()

+--------------+---------------------+
|specialization|avg(consultation_fee)|
+--------------+---------------------+
|     Neurology|               2000.0|
|   Dermatology|                825.0|
|    Cardiology|               1250.0|
|    Pediatrics|                900.0|
|   Orthopedics|               1450.0|
+--------------+---------------------+



Section 6 — Joins

In [65]:
patients.join(appointments,"patient_id").show()

+----------+------+---------+---+------+-----------------+--------------+---------+----------------+---------+
|patient_id|  name|     city|age|gender|registration_date|appointment_id|doctor_id|appointment_date|   status|
+----------+------+---------+---+------+-----------------+--------------+---------+----------------+---------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|             1|      101|      2024-03-01|Completed|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|             2|      102|      2024-03-01|Completed|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|             3|      103|      2024-03-02|Completed|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|             4|      104|      2024-03-02|  Pending|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|             5|      106|      2024-03-03|Completed|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|             6|      105|      2024-03-03|Completed|
|

In [66]:
patients.join(appointments,"patient_id")\
.select("name","city","appointment_date","status").show()

+------+---------+----------------+---------+
|  name|     city|appointment_date|   status|
+------+---------+----------------+---------+
| Aarav|Hyderabad|      2024-03-01|Completed|
| Priya|Bangalore|      2024-03-01|Completed|
| Rahul|   Mumbai|      2024-03-02|Completed|
| Sneha|    Delhi|      2024-03-02|  Pending|
| Kiran|  Chennai|      2024-03-03|Completed|
| Meera|Hyderabad|      2024-03-03|Completed|
|  Amit|     Pune|      2024-03-04|Cancelled|
|  Neha|    Delhi|      2024-03-04|Completed|
| Divya|Bangalore|      2024-03-05|Completed|
|Vikram|   Mumbai|      2024-03-05|Completed|
|Farhan|Hyderabad|      2024-03-06|  Pending|
|Simran|    Delhi|      2024-03-06|Completed|
| Aarav|Hyderabad|      2024-03-07|Completed|
| Rahul|   Mumbai|      2024-03-07|Completed|
| Meera|Hyderabad|      2024-03-08|Cancelled|
| Divya|Bangalore|      2024-03-08|Completed|
+------+---------+----------------+---------+



In [67]:
doctors.join(appointments,"doctor_id").show()

+---------+-----------+--------------+---------+----------------+--------------+----------+----------------+---------+
|doctor_id|doctor_name|specialization|     city|consultation_fee|appointment_id|patient_id|appointment_date|   status|
+---------+-----------+--------------+---------+----------------+--------------+----------+----------------+---------+
|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|             1|         1|      2024-03-01|Completed|
|      102|    Dr Iyer|   Dermatology|Bangalore|             800|             2|         2|      2024-03-01|Completed|
|      103|    Dr Khan|   Orthopedics|   Mumbai|            1500|             3|         3|      2024-03-02|Completed|
|      104|   Dr Reddy|    Pediatrics|    Delhi|             900|             4|         4|      2024-03-02|  Pending|
|      106|    Dr Nair|    Cardiology|  Chennai|            1300|             5|         5|      2024-03-03|Completed|
|      105|   Dr Mehta|     Neurology|Hyderabad|

In [69]:
doctors.join(appointments,"doctor_id")\
.select("doctor_name","specialization","appointment_date","status").show()

+-----------+--------------+----------------+---------+
|doctor_name|specialization|appointment_date|   status|
+-----------+--------------+----------------+---------+
|  Dr Sharma|    Cardiology|      2024-03-01|Completed|
|    Dr Iyer|   Dermatology|      2024-03-01|Completed|
|    Dr Khan|   Orthopedics|      2024-03-02|Completed|
|   Dr Reddy|    Pediatrics|      2024-03-02|  Pending|
|    Dr Nair|    Cardiology|      2024-03-03|Completed|
|   Dr Mehta|     Neurology|      2024-03-03|Completed|
|   Dr Verma|   Dermatology|      2024-03-04|Cancelled|
|     Dr Rao|   Orthopedics|      2024-03-04|Completed|
|    Dr Iyer|   Dermatology|      2024-03-05|Completed|
|    Dr Khan|   Orthopedics|      2024-03-05|Completed|
|  Dr Sharma|    Cardiology|      2024-03-06|  Pending|
|   Dr Reddy|    Pediatrics|      2024-03-06|Completed|
|   Dr Mehta|     Neurology|      2024-03-07|Completed|
|     Dr Rao|   Orthopedics|      2024-03-07|Completed|
|  Dr Sharma|    Cardiology|      2024-03-08|Can

In [70]:
appointments.join(bills,"appointment_id").show()

+--------------+----------+---------+----------------+---------+-------+-----------+------------+--------------+
|appointment_id|patient_id|doctor_id|appointment_date|   status|bill_id|bill_amount|payment_mode|payment_status|
+--------------+----------+---------+----------------+---------+-------+-----------+------------+--------------+
|             1|         1|      101|      2024-03-01|Completed|      1|       1200|         UPI|          Paid|
|             2|         2|      102|      2024-03-01|Completed|      2|        800| Credit Card|          Paid|
|             3|         3|      103|      2024-03-02|Completed|      3|       1500|        Cash|          Paid|
|             4|         4|      104|      2024-03-02|  Pending|      4|        900|         UPI|       Pending|
|             5|         5|      106|      2024-03-03|Completed|      5|       1300|  Debit Card|          Paid|
|             6|         6|      105|      2024-03-03|Completed|      6|       2000| Credit Card

In [71]:
appointments.join(bills,"appointment_id")\
.select("appointment_id","status","bill_amount","payment_status").show()

+--------------+---------+-----------+--------------+
|appointment_id|   status|bill_amount|payment_status|
+--------------+---------+-----------+--------------+
|             1|Completed|       1200|          Paid|
|             2|Completed|        800|          Paid|
|             3|Completed|       1500|          Paid|
|             4|  Pending|        900|       Pending|
|             5|Completed|       1300|          Paid|
|             6|Completed|       2000|          Paid|
|             7|Cancelled|        850|     Cancelled|
|             8|Completed|       1400|          Paid|
|             9|Completed|        800|          Paid|
|            10|Completed|       1500|          Paid|
|            11|  Pending|       1200|       Pending|
|            12|Completed|        900|          Paid|
|            13|Completed|       2000|          Paid|
|            14|Completed|       1400|          Paid|
|            15|Cancelled|       1200|     Cancelled|
|            16|Completed|  

In [72]:
patients.join(appointments,"patient_id")\
.join(doctors,"doctor_id").show()

+---------+----------+------+---------+---+------+-----------------+--------------+----------------+---------+-----------+--------------+---------+----------------+
|doctor_id|patient_id|  name|     city|age|gender|registration_date|appointment_id|appointment_date|   status|doctor_name|specialization|     city|consultation_fee|
+---------+----------+------+---------+---+------+-----------------+--------------+----------------+---------+-----------+--------------+---------+----------------+
|      101|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|             1|      2024-03-01|Completed|  Dr Sharma|    Cardiology|Hyderabad|            1200|
|      102|         2| Priya|Bangalore| 34|Female|       2023-02-12|             2|      2024-03-01|Completed|    Dr Iyer|   Dermatology|Bangalore|             800|
|      103|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|             3|      2024-03-02|Completed|    Dr Khan|   Orthopedics|   Mumbai|            1500|
|      104

In [73]:
patients.join(appointments,"patient_id")\
.join(doctors,"doctor_id")\
.select("name","doctor_name","specialization","appointment_date").show()

+------+-----------+--------------+----------------+
|  name|doctor_name|specialization|appointment_date|
+------+-----------+--------------+----------------+
| Aarav|  Dr Sharma|    Cardiology|      2024-03-01|
| Priya|    Dr Iyer|   Dermatology|      2024-03-01|
| Rahul|    Dr Khan|   Orthopedics|      2024-03-02|
| Sneha|   Dr Reddy|    Pediatrics|      2024-03-02|
| Kiran|    Dr Nair|    Cardiology|      2024-03-03|
| Meera|   Dr Mehta|     Neurology|      2024-03-03|
|  Amit|   Dr Verma|   Dermatology|      2024-03-04|
|  Neha|     Dr Rao|   Orthopedics|      2024-03-04|
| Divya|    Dr Iyer|   Dermatology|      2024-03-05|
|Vikram|    Dr Khan|   Orthopedics|      2024-03-05|
|Farhan|  Dr Sharma|    Cardiology|      2024-03-06|
|Simran|   Dr Reddy|    Pediatrics|      2024-03-06|
| Aarav|   Dr Mehta|     Neurology|      2024-03-07|
| Rahul|     Dr Rao|   Orthopedics|      2024-03-07|
| Meera|  Dr Sharma|    Cardiology|      2024-03-08|
| Divya|    Dr Nair|    Cardiology|      2024-

In [74]:
patients.join(appointments,"patient_id")\
.join(doctors,"doctor_id")\
.join(bills,"appointment_id").show()

+--------------+---------+----------+------+---------+---+------+-----------------+----------------+---------+-----------+--------------+---------+----------------+-------+-----------+------------+--------------+
|appointment_id|doctor_id|patient_id|  name|     city|age|gender|registration_date|appointment_date|   status|doctor_name|specialization|     city|consultation_fee|bill_id|bill_amount|payment_mode|payment_status|
+--------------+---------+----------+------+---------+---+------+-----------------+----------------+---------+-----------+--------------+---------+----------------+-------+-----------+------------+--------------+
|             1|      101|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|      2024-03-01|Completed|  Dr Sharma|    Cardiology|Hyderabad|            1200|      1|       1200|         UPI|          Paid|
|             2|      102|         2| Priya|Bangalore| 34|Female|       2023-02-12|      2024-03-01|Completed|    Dr Iyer|   Dermatology|Bangalore| 

In [75]:
patients.join(appointments,"patient_id")\
.join(doctors,"doctor_id")\
.join(bills,"appointment_id")\
.select("name","doctor_name","status","payment_mode").show()

+------+-----------+---------+------------+
|  name|doctor_name|   status|payment_mode|
+------+-----------+---------+------------+
| Aarav|  Dr Sharma|Completed|         UPI|
| Priya|    Dr Iyer|Completed| Credit Card|
| Rahul|    Dr Khan|Completed|        Cash|
| Sneha|   Dr Reddy|  Pending|         UPI|
| Kiran|    Dr Nair|Completed|  Debit Card|
| Meera|   Dr Mehta|Completed| Credit Card|
|  Amit|   Dr Verma|Cancelled|        Cash|
|  Neha|     Dr Rao|Completed|         UPI|
| Divya|    Dr Iyer|Completed|         UPI|
|Vikram|    Dr Khan|Completed| Credit Card|
|Farhan|  Dr Sharma|  Pending|         UPI|
|Simran|   Dr Reddy|Completed|        Cash|
| Aarav|   Dr Mehta|Completed| Credit Card|
| Rahul|     Dr Rao|Completed|         UPI|
| Meera|  Dr Sharma|Cancelled|        Cash|
| Divya|    Dr Nair|Completed|  Debit Card|
+------+-----------+---------+------------+



Section 7 — Updating Data with withColumn

In [87]:
from pyspark.sql.functions import when,col, lit,concat
patients.withColumn("age_group",
    when(col("age")<30,"Young")
    .when(col("age")<50,"Adult")
    .otherwise("Senior")).show()

+----------+------+---------+---+------+-----------------+---------+
|patient_id|  name|     city|age|gender|registration_date|age_group|
+----------+------+---------+---+------+-----------------+---------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|    Young|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|    Adult|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|    Adult|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|    Young|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|    Adult|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|    Adult|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|    Adult|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|    Young|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|    Adult|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|   Senior|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|    Adult|
|        12|Simran|    Delhi| 25|F

In [79]:
patients.withColumn("hospital_name" ,lit("BotCampus Hospital")).show()

+----------+------+---------+---+------+-----------------+------------------+
|patient_id|  name|     city|age|gender|registration_date|     hospital_name|
+----------+------+---------+---+------+-----------------+------------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|BotCampus Hospital|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|BotCampus Hospital|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|BotCampus Hospital|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|BotCampus Hospital|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|BotCampus Hospital|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|BotCampus Hospital|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|BotCampus Hospital|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|BotCampus Hospital|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|BotCampus Hospital|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|BotCam

In [80]:
doctors.withColumn("fee_with_tax", col( "consultation_fee") * 1.18).show()

+---------+-----------+--------------+---------+----------------+------------+
|doctor_id|doctor_name|specialization|     city|consultation_fee|fee_with_tax|
+---------+-----------+--------------+---------+----------------+------------+
|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|      1416.0|
|      102|    Dr Iyer|   Dermatology|Bangalore|             800|       944.0|
|      103|    Dr Khan|   Orthopedics|   Mumbai|            1500|      1770.0|
|      104|   Dr Reddy|    Pediatrics|    Delhi|             900|      1062.0|
|      105|   Dr Mehta|     Neurology|Hyderabad|            2000|      2360.0|
|      106|    Dr Nair|    Cardiology|  Chennai|            1300|      1534.0|
|      107|   Dr Verma|   Dermatology|     Pune|             850|      1003.0|
|      108|     Dr Rao|   Orthopedics|    Delhi|            1400|      1652.0|
+---------+-----------+--------------+---------+----------------+------------+



In [82]:
bills.withColumn("bill_with_tax", col("bill_amount") * 1.18 ).show()

+-------+--------------+-----------+------------+--------------+-------------+
|bill_id|appointment_id|bill_amount|payment_mode|payment_status|bill_with_tax|
+-------+--------------+-----------+------------+--------------+-------------+
|      1|             1|       1200|         UPI|          Paid|       1416.0|
|      2|             2|        800| Credit Card|          Paid|        944.0|
|      3|             3|       1500|        Cash|          Paid|       1770.0|
|      4|             4|        900|         UPI|       Pending|       1062.0|
|      5|             5|       1300|  Debit Card|          Paid|       1534.0|
|      6|             6|       2000| Credit Card|          Paid|       2360.0|
|      7|             7|        850|        Cash|     Cancelled|       1003.0|
|      8|             8|       1400|         UPI|          Paid|       1652.0|
|      9|             9|        800|         UPI|          Paid|        944.0|
|     10|            10|       1500| Credit Card|   

In [84]:
bills.withColumn("bill_in_thousands",col("bill_amount") / 1000).show()

+-------+--------------+-----------+------------+--------------+-----------------+
|bill_id|appointment_id|bill_amount|payment_mode|payment_status|bill_in_thousands|
+-------+--------------+-----------+------------+--------------+-----------------+
|      1|             1|       1200|         UPI|          Paid|              1.2|
|      2|             2|        800| Credit Card|          Paid|              0.8|
|      3|             3|       1500|        Cash|          Paid|              1.5|
|      4|             4|        900|         UPI|       Pending|              0.9|
|      5|             5|       1300|  Debit Card|          Paid|              1.3|
|      6|             6|       2000| Credit Card|          Paid|              2.0|
|      7|             7|        850|        Cash|     Cancelled|             0.85|
|      8|             8|       1400|         UPI|          Paid|              1.4|
|      9|             9|        800|         UPI|          Paid|              0.8|
|   

In [85]:
patients.withColumn("country",lit("India")).show()

+----------+------+---------+---+------+-----------------+-------+
|patient_id|  name|     city|age|gender|registration_date|country|
+----------+------+---------+---+------+-----------------+-------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|  India|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|  India|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|  India|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|  India|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|  India|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|  India|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|  India|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|  India|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|  India|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|  India|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|  India|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|  In

In [89]:
doctors.withColumn("doctor_label",concat(col("doctor_name"),lit(" + "),col("specialization"))).show()

+---------+-----------+--------------+---------+----------------+--------------------+
|doctor_id|doctor_name|specialization|     city|consultation_fee|        doctor_label|
+---------+-----------+--------------+---------+----------------+--------------------+
|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|Dr Sharma + Cardi...|
|      102|    Dr Iyer|   Dermatology|Bangalore|             800|Dr Iyer + Dermato...|
|      103|    Dr Khan|   Orthopedics|   Mumbai|            1500|Dr Khan + Orthope...|
|      104|   Dr Reddy|    Pediatrics|    Delhi|             900|Dr Reddy + Pediat...|
|      105|   Dr Mehta|     Neurology|Hyderabad|            2000|Dr Mehta + Neurology|
|      106|    Dr Nair|    Cardiology|  Chennai|            1300|Dr Nair + Cardiology|
|      107|   Dr Verma|   Dermatology|     Pune|             850|Dr Verma + Dermat...|
|      108|     Dr Rao|   Orthopedics|    Delhi|            1400|Dr Rao + Orthopedics|
+---------+-----------+--------------+-----

In [91]:
patients.withColumn("patient_label",concat(col("name"),lit(" + "),col("city"))).show()

+----------+------+---------+---+------+-----------------+------------------+
|patient_id|  name|     city|age|gender|registration_date|     patient_label|
+----------+------+---------+---+------+-----------------+------------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10| Aarav + Hyderabad|
|         2| Priya|Bangalore| 34|Female|       2023-02-12| Priya + Bangalore|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|    Rahul + Mumbai|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|     Sneha + Delhi|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|   Kiran + Chennai|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10| Meera + Hyderabad|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|       Amit + Pune|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|      Neha + Delhi|
|         9| Divya|Bangalore| 33|Female|       2023-07-15| Divya + Bangalore|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|   Vik

In [92]:
bills.withColumn("high_bill_flag",col("bill_amount")>1500).show()

+-------+--------------+-----------+------------+--------------+--------------+
|bill_id|appointment_id|bill_amount|payment_mode|payment_status|high_bill_flag|
+-------+--------------+-----------+------------+--------------+--------------+
|      1|             1|       1200|         UPI|          Paid|         false|
|      2|             2|        800| Credit Card|          Paid|         false|
|      3|             3|       1500|        Cash|          Paid|         false|
|      4|             4|        900|         UPI|       Pending|         false|
|      5|             5|       1300|  Debit Card|          Paid|         false|
|      6|             6|       2000| Credit Card|          Paid|          true|
|      7|             7|        850|        Cash|     Cancelled|         false|
|      8|             8|       1400|         UPI|          Paid|         false|
|      9|             9|        800|         UPI|          Paid|         false|
|     10|            10|       1500| Cre

In [93]:
patients.withColumn("senior_patient_flag",col("age")>40).show()

+----------+------+---------+---+------+-----------------+-------------------+
|patient_id|  name|     city|age|gender|registration_date|senior_patient_flag|
+----------+------+---------+---+------+-----------------+-------------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|              false|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|              false|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|               true|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|              false|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|              false|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|              false|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|               true|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|              false|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|              false|
|        10|Vikram|   Mumbai| 52|  Male|       2023-

Section 8 — When / Otherwise

In [94]:
patients.withColumn("age_group",
    when(col("age")<30,"Young")
    .when(col("age")<50,"Adult")
    .otherwise("Senior")).show()

+----------+------+---------+---+------+-----------------+---------+
|patient_id|  name|     city|age|gender|registration_date|age_group|
+----------+------+---------+---+------+-----------------+---------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|    Young|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|    Adult|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|    Adult|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|    Young|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|    Adult|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|    Adult|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|    Adult|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|    Young|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|    Adult|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|   Senior|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|    Adult|
|        12|Simran|    Delhi| 25|F

In [96]:
bills.withColumn("bill_category",
    when(col("bill_amount")>1500,"high")
    .when(col("bill_amount")>1000,"medium")
    .otherwise("low")).show()

+-------+--------------+-----------+------------+--------------+-------------+
|bill_id|appointment_id|bill_amount|payment_mode|payment_status|bill_category|
+-------+--------------+-----------+------------+--------------+-------------+
|      1|             1|       1200|         UPI|          Paid|       medium|
|      2|             2|        800| Credit Card|          Paid|          low|
|      3|             3|       1500|        Cash|          Paid|       medium|
|      4|             4|        900|         UPI|       Pending|          low|
|      5|             5|       1300|  Debit Card|          Paid|       medium|
|      6|             6|       2000| Credit Card|          Paid|         high|
|      7|             7|        850|        Cash|     Cancelled|          low|
|      8|             8|       1400|         UPI|          Paid|       medium|
|      9|             9|        800|         UPI|          Paid|          low|
|     10|            10|       1500| Credit Card|   

In [97]:
doctors.withColumn("fee_category",
    when(col("consultation_fee")>1500,"Premium")
    .when(col("consultation_fee")>1000,"Standard")
    .otherwise("Basic")).show()

+---------+-----------+--------------+---------+----------------+------------+
|doctor_id|doctor_name|specialization|     city|consultation_fee|fee_category|
+---------+-----------+--------------+---------+----------------+------------+
|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|    Standard|
|      102|    Dr Iyer|   Dermatology|Bangalore|             800|       Basic|
|      103|    Dr Khan|   Orthopedics|   Mumbai|            1500|    Standard|
|      104|   Dr Reddy|    Pediatrics|    Delhi|             900|       Basic|
|      105|   Dr Mehta|     Neurology|Hyderabad|            2000|     Premium|
|      106|    Dr Nair|    Cardiology|  Chennai|            1300|    Standard|
|      107|   Dr Verma|   Dermatology|     Pune|             850|       Basic|
|      108|     Dr Rao|   Orthopedics|    Delhi|            1400|    Standard|
+---------+-----------+--------------+---------+----------------+------------+



In [98]:
appointments.withColumn("priority",
    when(col("status")=="Pending","High")
    .when(col("status")=="Completed","Normal")
    .otherwise("Low")).show()



+--------------+----------+---------+----------------+---------+--------+
|appointment_id|patient_id|doctor_id|appointment_date|   status|priority|
+--------------+----------+---------+----------------+---------+--------+
|             1|         1|      101|      2024-03-01|Completed|  Normal|
|             2|         2|      102|      2024-03-01|Completed|  Normal|
|             3|         3|      103|      2024-03-02|Completed|  Normal|
|             4|         4|      104|      2024-03-02|  Pending|    High|
|             5|         5|      106|      2024-03-03|Completed|  Normal|
|             6|         6|      105|      2024-03-03|Completed|  Normal|
|             7|         7|      107|      2024-03-04|Cancelled|     Low|
|             8|         8|      108|      2024-03-04|Completed|  Normal|
|             9|         9|      102|      2024-03-05|Completed|  Normal|
|            10|        10|      103|      2024-03-05|Completed|  Normal|
|            11|        11|      101| 

In [99]:
patients.withColumn("zone",
    when(col("city").isin("Hyderabad","Bangalore","Chennai"),"South")
    .otherwise("Metro")).show()

+----------+------+---------+---+------+-----------------+-----+
|patient_id|  name|     city|age|gender|registration_date| zone|
+----------+------+---------+---+------+-----------------+-----+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|South|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|South|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|Metro|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|Metro|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|South|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|South|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|Metro|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|Metro|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|South|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|Metro|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|South|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|Metro|
+----------+------+------

Section 9 — Date Functions

In [100]:
from pyspark.sql.functions import to_date,year,month,datediff,current_date
patients.withColumn("registration_date",to_date(col("registration_date"))).show()

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|
+----------+------+---------+---+------+-----------------+



In [102]:
patients.withColumn("year",year("registration_date")).show()

+----------+------+---------+---+------+-----------------+----+
|patient_id|  name|     city|age|gender|registration_date|year|
+----------+------+---------+---+------+-----------------+----+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|2023|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|2023|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|2023|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|2023|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|2023|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|2023|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|2023|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|2023|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|2023|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|2023|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|2023|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|2023|
+----------+------+---------+---+------+

In [104]:
patients.withColumn("month",month("registration_date")).show()

+----------+------+---------+---+------+-----------------+-----+
|patient_id|  name|     city|age|gender|registration_date|month|
+----------+------+---------+---+------+-----------------+-----+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|    1|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|    2|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|    3|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|    4|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|    5|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|    6|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|    6|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|    7|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|    7|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|    8|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|    8|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|    8|
+----------+------+------

In [105]:
appointments.withColumn("appointment_date",to_date(col("appointment_date"))).show()

+--------------+----------+---------+----------------+---------+
|appointment_id|patient_id|doctor_id|appointment_date|   status|
+--------------+----------+---------+----------------+---------+
|             1|         1|      101|      2024-03-01|Completed|
|             2|         2|      102|      2024-03-01|Completed|
|             3|         3|      103|      2024-03-02|Completed|
|             4|         4|      104|      2024-03-02|  Pending|
|             5|         5|      106|      2024-03-03|Completed|
|             6|         6|      105|      2024-03-03|Completed|
|             7|         7|      107|      2024-03-04|Cancelled|
|             8|         8|      108|      2024-03-04|Completed|
|             9|         9|      102|      2024-03-05|Completed|
|            10|        10|      103|      2024-03-05|Completed|
|            11|        11|      101|      2024-03-06|  Pending|
|            12|        12|      104|      2024-03-06|Completed|
|            13|         

In [109]:
appointments.withColumn("month",month("appointment_date")).show()

+--------------+----------+---------+----------------+---------+-----+
|appointment_id|patient_id|doctor_id|appointment_date|   status|month|
+--------------+----------+---------+----------------+---------+-----+
|             1|         1|      101|      2024-03-01|Completed|    3|
|             2|         2|      102|      2024-03-01|Completed|    3|
|             3|         3|      103|      2024-03-02|Completed|    3|
|             4|         4|      104|      2024-03-02|  Pending|    3|
|             5|         5|      106|      2024-03-03|Completed|    3|
|             6|         6|      105|      2024-03-03|Completed|    3|
|             7|         7|      107|      2024-03-04|Cancelled|    3|
|             8|         8|      108|      2024-03-04|Completed|    3|
|             9|         9|      102|      2024-03-05|Completed|    3|
|            10|        10|      103|      2024-03-05|Completed|    3|
|            11|        11|      101|      2024-03-06|  Pending|    3|
|     

In [110]:
appointments.groupBy("appointment_date").count().show()

+----------------+-----+
|appointment_date|count|
+----------------+-----+
|      2024-03-05|    2|
|      2024-03-06|    2|
|      2024-03-08|    2|
|      2024-03-04|    2|
|      2024-03-02|    2|
|      2024-03-01|    2|
|      2024-03-03|    2|
|      2024-03-07|    2|
+----------------+-----+



In [111]:
appointments.groupBy(month("appointment_date")).count().show()

+-----------------------+-----+
|month(appointment_date)|count|
+-----------------------+-----+
|                      3|   16|
+-----------------------+-----+



In [112]:

patients.filter(col("registration_date")>"2023-06-01").show()



+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|
+----------+------+---------+---+------+-----------------+



In [113]:
patients.withColumn("days_since_reg",
    datediff(current_date(),"registration_date")).show()

appointments.withColumn("days_since_app",
    datediff(current_date(),"appointment_date")).show()

+----------+------+---------+---+------+-----------------+--------------+
|patient_id|  name|     city|age|gender|registration_date|days_since_reg|
+----------+------+---------+---+------+-----------------+--------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|          1203|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|          1170|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|          1140|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|          1108|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|          1082|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|          1052|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|          1040|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|          1022|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|          1017|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|          1000|
|        11|Farhan|Hyderabad| 39|  Mal

Section 10 — Window Functions

In [116]:
patients.createOrReplaceTempView("patients")
doctors.createOrReplaceTempView("doctors")
appointments.createOrReplaceTempView("appointments")
bills.createOrReplaceTempView("bills")

In [117]:
spark.sql("""
SELECT *,
RANK() OVER (PARTITION BY city ORDER BY age DESC) AS rank
FROM patients
""").show()

+----------+------+---------+---+------+-----------------+----+
|patient_id|  name|     city|age|gender|registration_date|rank|
+----------+------+---------+---+------+-----------------+----+
|         2| Priya|Bangalore| 34|Female|       2023-02-12|   1|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|   2|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|   1|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|   1|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|   2|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|   3|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|   1|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|   2|
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|   3|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|   1|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|   2|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|   1|
+----------+------+---------+---+------+

In [118]:

spark.sql("""
SELECT *
FROM (
    SELECT *,
    ROW_NUMBER() OVER (PARTITION BY city ORDER BY age DESC) AS rn
    FROM patients
) t
WHERE rn = 1
""").show()

+----------+------+---------+---+------+-----------------+---+
|patient_id|  name|     city|age|gender|registration_date| rn|
+----------+------+---------+---+------+-----------------+---+
|         2| Priya|Bangalore| 34|Female|       2023-02-12|  1|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|  1|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|  1|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|  1|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|  1|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|  1|
+----------+------+---------+---+------+-----------------+---+



In [119]:
spark.sql("""
SELECT *,
DENSE_RANK() OVER (PARTITION BY specialization ORDER BY consultation_fee DESC) AS dense_rank
FROM doctors
""").show()

+---------+-----------+--------------+---------+----------------+----------+
|doctor_id|doctor_name|specialization|     city|consultation_fee|dense_rank|
+---------+-----------+--------------+---------+----------------+----------+
|      106|    Dr Nair|    Cardiology|  Chennai|            1300|         1|
|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|         2|
|      107|   Dr Verma|   Dermatology|     Pune|             850|         1|
|      102|    Dr Iyer|   Dermatology|Bangalore|             800|         2|
|      105|   Dr Mehta|     Neurology|Hyderabad|            2000|         1|
|      103|    Dr Khan|   Orthopedics|   Mumbai|            1500|         1|
|      108|     Dr Rao|   Orthopedics|    Delhi|            1400|         2|
|      104|   Dr Reddy|    Pediatrics|    Delhi|             900|         1|
+---------+-----------+--------------+---------+----------------+----------+



In [120]:
spark.sql("""
SELECT *
FROM (
    SELECT *,
    RANK() OVER (PARTITION BY city ORDER BY age DESC) AS rnk
    FROM patients
) t
WHERE rnk <= 2
""").show()

+----------+------+---------+---+------+-----------------+---+
|patient_id|  name|     city|age|gender|registration_date|rnk|
+----------+------+---------+---+------+-----------------+---+
|         2| Priya|Bangalore| 34|Female|       2023-02-12|  1|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|  2|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|  1|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|  1|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|  2|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|  1|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|  2|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|  1|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|  2|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|  1|
+----------+------+---------+---+------+-----------------+---+



In [121]:
spark.sql("""
SELECT *,
RANK() OVER (ORDER BY consultation_fee DESC) AS rank
FROM doctors
""").show()

+---------+-----------+--------------+---------+----------------+----+
|doctor_id|doctor_name|specialization|     city|consultation_fee|rank|
+---------+-----------+--------------+---------+----------------+----+
|      105|   Dr Mehta|     Neurology|Hyderabad|            2000|   1|
|      103|    Dr Khan|   Orthopedics|   Mumbai|            1500|   2|
|      108|     Dr Rao|   Orthopedics|    Delhi|            1400|   3|
|      106|    Dr Nair|    Cardiology|  Chennai|            1300|   4|
|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|   5|
|      104|   Dr Reddy|    Pediatrics|    Delhi|             900|   6|
|      107|   Dr Verma|   Dermatology|     Pune|             850|   7|
|      102|    Dr Iyer|   Dermatology|Bangalore|             800|   8|
+---------+-----------+--------------+---------+----------------+----+



In [122]:
spark.sql("""
SELECT *,
SUM(bill_amount) OVER (
    PARTITION BY payment_mode
    ORDER BY bill_id
) AS running_total
FROM bills
""").show()

+-------+--------------+-----------+------------+--------------+-------------+
|bill_id|appointment_id|bill_amount|payment_mode|payment_status|running_total|
+-------+--------------+-----------+------------+--------------+-------------+
|      3|             3|       1500|        Cash|          Paid|         1500|
|      7|             7|        850|        Cash|     Cancelled|         2350|
|     12|            12|        900|        Cash|          Paid|         3250|
|     15|            15|       1200|        Cash|     Cancelled|         4450|
|      2|             2|        800| Credit Card|          Paid|          800|
|      6|             6|       2000| Credit Card|          Paid|         2800|
|     10|            10|       1500| Credit Card|          Paid|         4300|
|     13|            13|       2000| Credit Card|          Paid|         6300|
|      5|             5|       1300|  Debit Card|          Paid|         1300|
|     16|            16|       1300|  Debit Card|   

In [123]:
spark.sql("""
SELECT *,
COUNT(appointment_id) OVER (
    PARTITION BY doctor_id
    ORDER BY appointment_date
) AS running_count
FROM appointments
""").show()

+--------------+----------+---------+----------------+---------+-------------+
|appointment_id|patient_id|doctor_id|appointment_date|   status|running_count|
+--------------+----------+---------+----------------+---------+-------------+
|             1|         1|      101|      2024-03-01|Completed|            1|
|            11|        11|      101|      2024-03-06|  Pending|            2|
|            15|         6|      101|      2024-03-08|Cancelled|            3|
|             2|         2|      102|      2024-03-01|Completed|            1|
|             9|         9|      102|      2024-03-05|Completed|            2|
|             3|         3|      103|      2024-03-02|Completed|            1|
|            10|        10|      103|      2024-03-05|Completed|            2|
|             4|         4|      104|      2024-03-02|  Pending|            1|
|            12|        12|      104|      2024-03-06|Completed|            2|
|             6|         6|      105|      2024-03-0

In [124]:
spark.sql("""
SELECT *,
RANK() OVER (ORDER BY patient_count DESC) AS rank
FROM (
    SELECT city, COUNT(*) AS patient_count
    FROM patients
    GROUP BY city
) t
""").show()

+---------+-------------+----+
|     city|patient_count|rank|
+---------+-------------+----+
|    Delhi|            3|   1|
|Hyderabad|            3|   1|
|Bangalore|            2|   3|
|   Mumbai|            2|   3|
|  Chennai|            1|   5|
|     Pune|            1|   5|
+---------+-------------+----+



Section 11 — RDD Exercises

In [125]:
logs = sc.textFile("hospital_logs.txt")

In [126]:
logs.count()

20

In [127]:
logs.map(lambda x:x.split()[0]).collect()

['Aarav',
 'Priya',
 'Rahul',
 'Sneha',
 'Aarav',
 'Kiran',
 'Meera',
 'Vikram',
 'Divya',
 'Farhan',
 'Simran',
 'Neha',
 'Amit',
 'Rahul',
 'Meera',
 'Aarav',
 'Priya',
 'Divya',
 'Vikram',
 'Farhan']

In [128]:
logs.map(lambda x:x.split()[1]).collect()

['login',
 'login',
 'appointment',
 'login',
 'payment',
 'appointment',
 'login',
 'appointment',
 'payment',
 'login',
 'appointment',
 'payment',
 'login',
 'payment',
 'appointment',
 'logout',
 'payment',
 'login',
 'payment',
 'appointment']

In [129]:
logs.map(lambda x:x.split()[0]).distinct().collect()

['Kiran',
 'Vikram',
 'Divya',
 'Simran',
 'Amit',
 'Aarav',
 'Priya',
 'Rahul',
 'Sneha',
 'Meera',
 'Farhan',
 'Neha']

In [130]:
logs.map(lambda x: (x.split()[1],1))\
.reduceByKey(lambda a,b:a+b).collect()

[('login', 7), ('payment', 6), ('appointment', 6), ('logout', 1)]

Bonus: Complex JSON Exercises

In [131]:
profiles.printSchema()


root
 |-- allergies: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- name: string (nullable = true)
 |-- patient_id: long (nullable = true)



In [132]:
profiles.select("patient_id","name","contact.phone","contact.email").show()

+----------+------+----------+---------------+
|patient_id|  name|     phone|          email|
+----------+------+----------+---------------+
|         1| Aarav|9000011111| aarav@mail.com|
|         2| Priya|9000022222| priya@mail.com|
|         3| Rahul|9000033333| rahul@mail.com|
|         6| Meera|9000066666| meera@mail.com|
|        10|Vikram|9000101010|vikram@mail.com|
+----------+------+----------+---------------+



In [134]:
from pyspark.sql.functions import explode
profiles.select("patient_id","name",explode("allergies")).show()


+----------+------+-------+
|patient_id|  name|    col|
+----------+------+-------+
|         1| Aarav|   Dust|
|         1| Aarav|Peanuts|
|         2| Priya| Pollen|
|         3| Rahul|   Dust|
|         3| Rahul|   Milk|
|         6| Meera|Seafood|
|        10|Vikram| Pollen|
|        10|Vikram|   Dust|
+----------+------+-------+



In [135]:
profiles.select("patient_id","name",explode("allergies")).show()


+----------+------+-------+
|patient_id|  name|    col|
+----------+------+-------+
|         1| Aarav|   Dust|
|         1| Aarav|Peanuts|
|         2| Priya| Pollen|
|         3| Rahul|   Dust|
|         3| Rahul|   Milk|
|         6| Meera|Seafood|
|        10|Vikram| Pollen|
|        10|Vikram|   Dust|
+----------+------+-------+



In [138]:
from pyspark.sql.functions import array_contains
profiles.filter(array_contains("allergies","Dust")).show()

+---------------+--------------------+------+----------+
|      allergies|             contact|  name|patient_id|
+---------------+--------------------+------+----------+
|[Dust, Peanuts]|{aarav@mail.com, ...| Aarav|         1|
|   [Dust, Milk]|{rahul@mail.com, ...| Rahul|         3|
| [Pollen, Dust]|{vikram@mail.com,...|Vikram|        10|
+---------------+--------------------+------+----------+



In [139]:
profiles.select(explode("allergies")).distinct().show()

profiles.join(patients,"patient_id").show()



+-------+
|    col|
+-------+
|   Milk|
|   Dust|
|Peanuts|
| Pollen|
|Seafood|
+-------+

+----------+---------------+--------------------+------+------+---------+---+------+-----------------+
|patient_id|      allergies|             contact|  name|  name|     city|age|gender|registration_date|
+----------+---------------+--------------------+------+------+---------+---+------+-----------------+
|         1|[Dust, Peanuts]|{aarav@mail.com, ...| Aarav| Aarav|Hyderabad| 29|  Male|       2023-01-10|
|         2|       [Pollen]|{priya@mail.com, ...| Priya| Priya|Bangalore| 34|Female|       2023-02-12|
|         3|   [Dust, Milk]|{rahul@mail.com, ...| Rahul| Rahul|   Mumbai| 41|  Male|       2023-03-14|
|         6|      [Seafood]|{meera@mail.com, ...| Meera| Meera|Hyderabad| 31|Female|       2023-06-10|
|        10| [Pollen, Dust]|{vikram@mail.com,...|Vikram|Vikram|   Mumbai| 52|  Male|       2023-08-01|
+----------+---------------+--------------------+------+------+---------+---+------+-

In [142]:
profiles.join(patients,"patient_id")\
.select(patients.name,patients.city,"contact.email",profiles.allergies).show()

+------+---------+---------------+---------------+
|  name|     city|          email|      allergies|
+------+---------+---------------+---------------+
| Aarav|Hyderabad| aarav@mail.com|[Dust, Peanuts]|
| Priya|Bangalore| priya@mail.com|       [Pollen]|
| Rahul|   Mumbai| rahul@mail.com|   [Dust, Milk]|
| Meera|Hyderabad| meera@mail.com|      [Seafood]|
|Vikram|   Mumbai|vikram@mail.com| [Pollen, Dust]|
+------+---------+---------------+---------------+



In [140]:
profiles.join(patients,"patient_id")\
.select("city",explode("allergies").alias("allergy"))\
.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    1|
|   Mumbai|    4|
|Hyderabad|    3|
+---------+-----+



In [143]:
patients.createOrReplaceTempView("patients")
doctors.createOrReplaceTempView("doctors")
appointments.createOrReplaceTempView("appointments")
bills.createOrReplaceTempView("bills")

In [144]:
spark.sql("SELECT * FROM patients").show()

+----------+------+---------+---+------+-----------------+
|patient_id|  name|     city|age|gender|registration_date|
+----------+------+---------+---+------+-----------------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|
|         6| Meera|Hyderabad| 31|Female|       2023-06-10|
|         7|  Amit|     Pune| 45|  Male|       2023-06-22|
|         8|  Neha|    Delhi| 28|Female|       2023-07-10|
|         9| Divya|Bangalore| 33|Female|       2023-07-15|
|        10|Vikram|   Mumbai| 52|  Male|       2023-08-01|
|        11|Farhan|Hyderabad| 39|  Male|       2023-08-10|
|        12|Simran|    Delhi| 25|Female|       2023-08-21|
+----------+------+---------+---+------+-----------------+



In [146]:
spark.sql("SELECT * FROM patients WHERE city='Hyderbad'").show()

+----------+----+----+---+------+-----------------+
|patient_id|name|city|age|gender|registration_date|
+----------+----+----+---+------+-----------------+
+----------+----+----+---+------+-----------------+



In [147]:
spark.sql("SELECT city,count(*) FROM patients GROUP BY city").show()

+---------+--------+
|     city|count(1)|
+---------+--------+
|Bangalore|       2|
|  Chennai|       1|
|   Mumbai|       2|
|     Pune|       1|
|    Delhi|       3|
|Hyderabad|       3|
+---------+--------+



In [148]:
spark.sql("SELECT specialization,count(*) FROM doctors GROUP BY specialization").show()

+--------------+--------+
|specialization|count(1)|
+--------------+--------+
|     Neurology|       1|
|   Dermatology|       2|
|    Cardiology|       2|
|    Pediatrics|       1|
|   Orthopedics|       2|
+--------------+--------+



In [149]:
spark.sql("SELECT * FROM patients p JOIN appointments a ON p.patient_id=a.patient_id").show()

+----------+------+---------+---+------+-----------------+--------------+----------+---------+----------------+---------+
|patient_id|  name|     city|age|gender|registration_date|appointment_id|patient_id|doctor_id|appointment_date|   status|
+----------+------+---------+---+------+-----------------+--------------+----------+---------+----------------+---------+
|         1| Aarav|Hyderabad| 29|  Male|       2023-01-10|             1|         1|      101|      2024-03-01|Completed|
|         2| Priya|Bangalore| 34|Female|       2023-02-12|             2|         2|      102|      2024-03-01|Completed|
|         3| Rahul|   Mumbai| 41|  Male|       2023-03-14|             3|         3|      103|      2024-03-02|Completed|
|         4| Sneha|    Delhi| 26|Female|       2023-04-15|             4|         4|      104|      2024-03-02|  Pending|
|         5| Kiran|  Chennai| 37|  Male|       2023-05-11|             5|         5|      106|      2024-03-03|Completed|
|         6| Meera|Hyder

In [152]:
spark.sql("SELECT * FROM doctors d JOIN appointments a ON d.doctor_id=a.doctor_id").show()

+---------+-----------+--------------+---------+----------------+--------------+----------+---------+----------------+---------+
|doctor_id|doctor_name|specialization|     city|consultation_fee|appointment_id|patient_id|doctor_id|appointment_date|   status|
+---------+-----------+--------------+---------+----------------+--------------+----------+---------+----------------+---------+
|      101|  Dr Sharma|    Cardiology|Hyderabad|            1200|             1|         1|      101|      2024-03-01|Completed|
|      102|    Dr Iyer|   Dermatology|Bangalore|             800|             2|         2|      102|      2024-03-01|Completed|
|      103|    Dr Khan|   Orthopedics|   Mumbai|            1500|             3|         3|      103|      2024-03-02|Completed|
|      104|   Dr Reddy|    Pediatrics|    Delhi|             900|             4|         4|      104|      2024-03-02|  Pending|
|      106|    Dr Nair|    Cardiology|  Chennai|            1300|             5|         5|      

In [153]:
spark.sql("SELECT * FROM  appointments a join bills b ON a.appointment_id=a.appointment_id").show()

+--------------+----------+---------+----------------+---------+-------+--------------+-----------+------------+--------------+
|appointment_id|patient_id|doctor_id|appointment_date|   status|bill_id|appointment_id|bill_amount|payment_mode|payment_status|
+--------------+----------+---------+----------------+---------+-------+--------------+-----------+------------+--------------+
|             1|         1|      101|      2024-03-01|Completed|      1|             1|       1200|         UPI|          Paid|
|             1|         1|      101|      2024-03-01|Completed|      2|             2|        800| Credit Card|          Paid|
|             1|         1|      101|      2024-03-01|Completed|      3|             3|       1500|        Cash|          Paid|
|             1|         1|      101|      2024-03-01|Completed|      4|             4|        900|         UPI|       Pending|
|             1|         1|      101|      2024-03-01|Completed|      5|             5|       1300|  Deb

In [154]:
spark.sql("SELECT payment_mode, SUM(bill_amount) FROM bills GROUP BY payment_mode").show()

+------------+----------------+
|payment_mode|sum(bill_amount)|
+------------+----------------+
| Credit Card|            6300|
|        Cash|            4450|
|  Debit Card|            2600|
|         UPI|            6900|
+------------+----------------+



In [155]:
spark.sql("""
SELECT doctor_name, consultation_fee,
RANK() OVER (ORDER BY consultation_fee DESC) AS rank
FROM doctors
""").show()



+-----------+----------------+----+
|doctor_name|consultation_fee|rank|
+-----------+----------------+----+
|   Dr Mehta|            2000|   1|
|    Dr Khan|            1500|   2|
|     Dr Rao|            1400|   3|
|    Dr Nair|            1300|   4|
|  Dr Sharma|            1200|   5|
|   Dr Reddy|             900|   6|
|   Dr Verma|             850|   7|
|    Dr Iyer|             800|   8|
+-----------+----------------+----+



In [156]:
spark.sql("""
SELECT p.name, SUM(b.bill_amount) total
FROM patients p
JOIN appointments a ON p.patient_id=a.patient_id
JOIN bills b ON a.appointment_id=b.appointment_id
GROUP BY p.name
ORDER BY total DESC
LIMIT 1
""").show()

+-----+-----+
| name|total|
+-----+-----+
|Meera| 3200|
+-----+-----+

